# Unit 11 — 文章要約: 抽出型から生成型へ

目安は **20〜25分**。これまでは入力を分類しました。今回は、複数レビューから短い要約を作ります。

この notebook は完全オフラインです。事前学習済み重みは取得しません。前半は NumPy の極小 decoder step、後半はインストール済み `transformers` の極小 T5 をランダム初期化し、同じ生成の仕組みをオフラインで動かします。

今日できるようになること:

1. 文分割 → TF-IDF → 文間 cosine similarity を組む
2. PageRank 風中心性と MMR で、重要かつ重複しない文を選ぶ
3. encoder-decoder の teacher forcing、shift right、`-100` mask を作る
4. greedy / beam / sampling と出力長コストを比較する
5. ROUGE-N / ROUGE-L、事実照合、品質・事実性・費用で方式を選ぶ

> C# なら、抽出型は既存要素を選ぶ `OrderByDescending(...).Take(k)`、生成型は状態を持つ decoder が `yield return` で次 token を1つずつ出す処理に近いです。

In [ ]:
# STEP 1: Unit 11 — 文章要約: 抽出型から生成型への処理を実行し、出力を照合する
from pathlib import Path
from collections import Counter
import re

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DATA_DIR = Path("data")
if not DATA_DIR.exists():
    DATA_DIR = Path("courses/kaggle-sprint/unit11-summarization/data")
if not DATA_DIR.exists():
    raise FileNotFoundError("unit11 またはリポジトリ直下から実行してください。")

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
print(f"data={DATA_DIR.resolve()}")
print(f"train={len(train_df)}, test={len(test_df)}, columns={train_df.columns.tolist()}")

def check(name, actual, expected, hint=""):
    import numpy as _np
    try:
        ok = actual is not None and bool(_np.all(_np.isclose(
            _np.asarray(actual, dtype=float), _np.asarray(expected, dtype=float)
        )))
    except (TypeError, ValueError):
        ok = actual == expected
    mark = "OK" if ok else "NG"
    if ok:
        print(f"[OK] {name}")
    else:
        detail = f"actual={actual!r} / expected={expected!r}"
        print(f"[NG] {name} — {detail}" + ("" if not hint else f" — ヒント: {hint}"))
    return ok

def safe_call(fn):
    try:
        return fn()
    except Exception:
        return None

def safe_get(mapping, key):
    return mapping.get(key) if isinstance(mapping, dict) else None

## 今日の流れ

各ブロックを **見る → 予測 → 変える → 書く → チェック** の順で進めます。

`document → sentences → vectors → similarity graph → selected sentences`

`source tokens → encoder → context → decoder token → decoder token → ...`

抽出型は原文の文をそのまま選ぶため、安価・監査可能・原理上は捏造しにくい方式です。まず抽出型で要求を満たせるか確認し、必要な品質差があるとき生成型や API を検討します。

## 1. 文を分け、文間類似度行列を作る

日本語の文分割は単純な `split("。")` だけでは、疑問符・感嘆符・改行・括弧内の句点を扱えません。ここでは教材データの規約に合わせ、終端記号 `。！？` または改行で区切ります。実務では文書形式ごとのテストケースを追加します。

`TfidfVectorizer` は文を TF-IDF 特徴行列へ変換する sklearn API、`cosine_similarity` は行ベクトル同士の cosine 類似度を返す API です。文が `n_sent` 個なら similarity shape は `(n_sent, n_sent)` です。

In [ ]:
# STEP 2: 1. 文を分け、文間類似度行列を作るの処理を実行し、出力を照合する
def reference_split_sentences(text):
    pieces = re.split(r"(?<=[。！？])\s*|\n+", str(text))
    return [piece.strip() for piece in pieces if piece.strip()]

document = train_df.loc[0, "document"]
sentences = reference_split_sentences(document)
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3))
sentence_tfidf = vectorizer.fit_transform(sentences)
sentence_similarity = cosine_similarity(sentence_tfidf)

print("sentences:", len(sentences))
print("TF-IDF shape:", sentence_tfidf.shape)
print("similarity shape:", sentence_similarity.shape)
print("first 3 sentences:")
for index, sentence in enumerate(sentences[:3]):
    print(index, sentence)

In [ ]:
# STEP 3: 1. 文を分け、文間類似度行列を作るの処理を実行し、出力を照合する
print("symmetric:", np.allclose(sentence_similarity, sentence_similarity.T))
print("diagonal:", np.round(np.diag(sentence_similarity), 3))
print("off-diagonal range:", round(float((sentence_similarity - np.eye(len(sentences))).min()), 3),
      round(float((sentence_similarity - np.eye(len(sentences))).max()), 3))

### 予測

12文を TF-IDF 化し、語彙特徴が450列になった場合の shape はどれでしょう。

A. `(12, 450)`　B. `(450, 12)`　C. `(12, 12)`

`(12, 12)` になるのは、その後に文同士を比較した similarity 行列です。

In [ ]:
# STEP 4: 予測の処理を実行し、出力を照合する
prediction_1 = "A"
print("答え:", prediction_1, "— 行が文、列がTF-IDF特徴です")

### 変える → 書く

`ngram_range=(2, 3)` を `(1, 2)` に変え、特徴数と類似度を見てください。細かい n-gram は表記揺れを拾いやすい一方、偶然の一致も増えます。

`split_ja_sentences` を完成させます。終端記号の直後または改行で分け、空文字を除きます。`re.split` は正規表現に一致する境界で文字列を分割する API です。

In [ ]:
# STEP 5: 変える → 書くの処理を実行し、出力を照合する
def split_ja_sentences(text):
    # TODO: 終端記号または改行で分割し、strip後の空文字を除く
    return None

split_example = "価格は良いです。配送も速いです！\nまた買います。"
learner_sentences = safe_call(lambda: split_ja_sentences(split_example))

In [ ]:
# STEP 6: 変える → 書くの処理を実行し、出力を照合する
check("sentence count", safe_call(lambda: len(learner_sentences)), 3, "3つの終端で分かれます")
check("sentence list", learner_sentences, ["価格は良いです。", "配送も速いです！", "また買います。"], "終端記号を文に残します")
check("first sentence", safe_call(lambda: learner_sentences[0]), "価格は良いです。", "strip した先頭文です")
check("no empty sentence", safe_call(lambda: all(learner_sentences)), True, "空文字を除きます")

## 2. PageRank 風中心性から MMR へ

文を node、類似度を edge weight とみなすと、似た文から支持される文は重要だと考えられます。PageRank 風のべき乗法では、行を確率へ正規化した transition 行列を作り、score を反復更新します。

中心性上位だけを取ると、同じ論点の言い換えが並びがちです。MMR (Maximal Marginal Relevance) は

`λ × relevance - (1 - λ) × 既選択文との最大類似度`

で、重要さと非冗長性を両立します。C# の greedy loop と同じで、1件選ぶたびに残候補の点を更新します。

In [ ]:
# STEP 7: 2. PageRank 風中心性から MMR への処理を実行し、出力を照合する
def pagerank_scores(similarity, damping=0.85, steps=50):
    graph = np.asarray(similarity, dtype=float).copy()
    np.fill_diagonal(graph, 0.0)
    row_sum = graph.sum(axis=1, keepdims=True)
    transition = np.divide(graph, row_sum, out=np.full_like(graph, 1 / len(graph)), where=row_sum > 0)
    scores = np.full(len(graph), 1 / len(graph))
    for _ in range(steps):
        scores = (1 - damping) / len(graph) + damping * transition.T @ scores
    return scores / scores.sum()

centrality = pagerank_scores(sentence_similarity)
centrality_top = np.argsort(-centrality)[:3]
print("centrality sum:", centrality.sum())
print("top indices:", centrality_top.tolist())
for index in centrality_top:
    print(round(float(centrality[index]), 4), sentences[index])

In [ ]:
# STEP 8: 2. PageRank 風中心性から MMR への処理を実行し、出力を照合する
def reference_mmr(relevance, similarity, k, lambda_value=0.7):
    selected = []
    remaining = list(range(len(relevance)))
    while remaining and len(selected) < k:
        if not selected:
            choice = max(remaining, key=lambda index: relevance[index])
        else:
            choice = max(remaining, key=lambda index: lambda_value * relevance[index] - (1 - lambda_value) * max(similarity[index, chosen] for chosen in selected))
        selected.append(choice)
        remaining.remove(choice)
    return selected

mmr_top = reference_mmr(centrality, sentence_similarity, k=min(3, len(sentences)), lambda_value=0.7)
print("MMR indices:", mmr_top)
for index in sorted(mmr_top):
    print(index, sentences[index])

measured = pd.DataFrame([
    ("中心性top-k", 1.283, 1.220, 0.0174, 0.4906),
    ("MMR λ=0.7", 1.797, 0.707, 0.0237, 0.6011),
], columns=["method", "covered_topics", "duplicates", "rouge2_abstract", "rouge2_extractive"])
measured

### 予測

`λ` を1.0にすると MMR は何を無視しますか。

A. relevance　B. 冗長性ペナルティ　C. 文数 k

式の `(1 - λ)` を見てください。

In [ ]:
# STEP 9: 予測の処理を実行し、出力を照合する
prediction_2 = "B"
print("答え:", prediction_2, "— λ=1は中心性だけのtop-kへ戻ります")

### 変える → 書く

`lambda_value` を 0.4 / 0.7 / 1.0 に変え、選ばれる文を比較してください。低すぎる λ は多様でも重要でない文を選びます。

docstring の実測では、MMR により重複が `1.220 → 0.707`、カバー論点が `1.283 → 1.797`、抽出型参照 ROUGE-2 が `0.4906 → 0.6011` になりました。

次の toy graph で `mmr_select` を完成させてください。

In [ ]:
# STEP 10: 変える → 書くの処理を実行し、出力を照合する
def mmr_select(relevance, similarity, k, lambda_value):
    # TODO: greedy に relevance - redundancy を最大化するindexを選ぶ
    return None

toy_relevance = np.array([0.90, 0.85, 0.70, 0.40])
toy_similarity = np.array([
    [1.00, 0.95, 0.10, 0.20],
    [0.95, 1.00, 0.20, 0.10],
    [0.10, 0.20, 1.00, 0.30],
    [0.20, 0.10, 0.30, 1.00],
])
learner_mmr = safe_call(lambda: mmr_select(toy_relevance, toy_similarity, 2, 0.7))

In [ ]:
# STEP 11: 変える → 書くの処理を実行し、出力を照合する
check("MMR first", safe_call(lambda: learner_mmr[0]), 0, "最初は relevance 最大です")
check("MMR selection", learner_mmr, [0, 2], "index 1 は0と似すぎるため2を選びます")
check("MMR length", safe_call(lambda: len(learner_mmr)), 2, "k件を選びます")
check("MMR unique", safe_call(lambda: len(set(learner_mmr)) == len(learner_mmr)), True, "選んだindexは remaining から除きます")

## 3. encoder-decoder と teacher forcing

encoder-only 分類器は入力を読んで class head を1回出します。encoder-decoder は、encoder が入力表現を作り、decoder が cross-attention でそれを参照しながら出力 token を順に書きます。

学習時の teacher forcing では正解 token を1位置右へずらし、decoder 入力にします。

- `labels`: 次に当てる正解 token
- `decoder_input_ids`: `[START] + labels[:-1]`
- pad位置の `labels`: `-100` にし、損失計算から除外

Hugging Face の seq2seq model は `model(input_ids=..., attention_mask=..., labels=...)` と `labels` を渡すと loss を返します。`-100` は PyTorch の cross entropy が既定で無視する値です。

In [ ]:
# STEP 12: 3. encoder-decoder と teacher forcingの処理を実行し、出力を照合する
rng_model = np.random.default_rng(11)
vocab_size, hidden_size = 10, 4
embedding = rng_model.normal(0, 0.2, size=(vocab_size, hidden_size))
output_weight = rng_model.normal(0, 0.3, size=(hidden_size, vocab_size))

source_ids = np.array([[2, 4, 5, 0]])
source_mask = (source_ids != 0).astype(float)
encoder_states = embedding[source_ids]
encoder_context = (encoder_states * source_mask[..., None]).sum(axis=1) / source_mask.sum(axis=1, keepdims=True)

def numpy_decoder_step(previous_token, context):
    hidden = np.tanh(embedding[previous_token] + context)
    return hidden @ output_weight

step_logits = numpy_decoder_step(previous_token=1, context=encoder_context[0])
print("encoder states:", encoder_states.shape)
print("encoder context:", encoder_context.shape)
print("decoder logits:", step_logits.shape)

In [ ]:
# STEP 13: 3. encoder-decoder と teacher forcingの処理を実行し、出力を照合する
labels_example = np.array([
    [4, 5, 6, 0, 0],
    [7, 8, 0, 0, 0],
])
loss_labels_reference = labels_example.copy()
loss_labels_reference[loss_labels_reference == 0] = -100
decoder_input_reference = np.full_like(labels_example, 0)
decoder_input_reference[:, 0] = 1
decoder_input_reference[:, 1:] = labels_example[:, :-1]

print("labels input:\n", labels_example)
print("decoder_input_ids:\n", decoder_input_reference)
print("labels for loss:\n", loss_labels_reference)
print("active loss tokens:", int((loss_labels_reference != -100).sum()))

### 予測

正解列が `[4, 5, 6, PAD, PAD]`、開始 token が1なら decoder 入力はどれでしょう。

A. `[4, 5, 6, 0, 0]`

B. `[1, 4, 5, 6, 0]`

C. `[1, 5, 6, 0, 0]`

In [ ]:
# STEP 14: 予測の処理を実行し、出力を照合する
prediction_3 = "B"
print("答え:", prediction_3, "— decoder は1つ前の正解を見て次を当てます")

### 変える → 書く

pad token を9に変える場合、値0を固定で探すコードは壊れます。必ず `pad_id` 引数を使ってください。

`prepare_teacher_forcing` を完成させ、decoder 入力と損失用 labels を返します。`labels` 原本は変更しないよう `copy()` します。

実 T5 の事前学習重みを使う場合は `AutoTokenizer.from_pretrained(...)` と `AutoModelForSeq2SeqLM.from_pretrained(...)` が取得と構築を行いますが、本教材ではネットワークアクセスを伴うため実行しません。

In [ ]:
# STEP 15: 変える → 書くの処理を実行し、出力を照合する
def prepare_teacher_forcing(labels, pad_id, decoder_start_id):
    # TODO: shift right した decoder_input_ids と -100 mask済み labels を返す
    return None

learner_teacher = safe_call(lambda: prepare_teacher_forcing(labels_example, 0, 1))

In [ ]:
# STEP 16: 変える → 書くの処理を実行し、出力を照合する
check("decoder input shape", safe_call(lambda: safe_get(learner_teacher, "decoder_input_ids").shape), (2, 5), "labels と同じshapeです")
check("shift-right row", safe_call(lambda: safe_get(learner_teacher, "decoder_input_ids")[0]), [1, 4, 5, 6, 0], "START + labels[:-1]")
check("loss mask row", safe_call(lambda: safe_get(learner_teacher, "loss_labels")[0]), [4, 5, 6, -100, -100], "pad を -100 にします")
check("active loss tokens", safe_call(lambda: int((safe_get(learner_teacher, "loss_labels") != -100).sum())), 5, "3 token + 2 token です")

## 4. greedy・beam・sampling と出力長コスト

推論時は正解がありません。直前までの生成列を入力へ戻し、EOS または `max_new_tokens` まで1 token ずつ進めます。

| 戦略 | 選び方 | 特徴 |
|---|---|---|
| greedy | 毎回最大logit | 高速・決定的、先読みしない |
| beam search | 上位系列を複数保持 | 品質候補↑、計算・memoryはbeam幅に比例 |
| sampling | 確率から抽選 | 多様、seed/temperature/top-k/top-pに依存 |

`model.generate(max_new_tokens=..., num_beams=...)` が実 API です。出力 token が1つ増えるたび decoder の前向き計算も1回増えるため、自前推論時間は概ね出力長に比例します。API料金でも出力 token は従量部分に加算されます。

In [ ]:
# STEP 17: 4. greedy・beam・sampling と出力長コストの処理を実行し、出力を照合する
BOS, TOKEN_A, TOKEN_B, EOS = 0, 1, 2, 3

def toy_next_logits(prefix):
    last = prefix[-1]
    table = {
        BOS: np.array([-9.0, 4.0, 2.0, -9.0]),
        TOKEN_A: np.array([-9.0, -9.0, 3.0, 1.0]),
        TOKEN_B: np.array([-9.0, -9.0, -9.0, 5.0]),
        EOS: np.array([-9.0, -9.0, -9.0, 5.0]),
    }
    return table[last]

def reference_greedy(step_function, start_id, eos_id, max_new_tokens):
    prefix, generated = [start_id], []
    for _ in range(max_new_tokens):
        token = int(np.argmax(step_function(prefix)))
        generated.append(token)
        prefix.append(token)
        if token == eos_id:
            break
    return generated

greedy_result = reference_greedy(toy_next_logits, BOS, EOS, 6)
print("greedy:", greedy_result)

In [ ]:
# STEP 18: 4. greedy・beam・sampling と出力長コストの処理を実行し、出力を照合する
def sample_decode(step_function, start_id, eos_id, max_new_tokens, temperature, seed):
    rng = np.random.default_rng(seed)
    prefix, generated = [start_id], []
    for _ in range(max_new_tokens):
        logits = step_function(prefix) / temperature
        probabilities = np.exp(logits - logits.max())
        probabilities /= probabilities.sum()
        token = int(rng.choice(len(probabilities), p=probabilities))
        generated.append(token)
        prefix.append(token)
        if token == eos_id:
            break
    return generated

for seed in [1, 2, 3]:
    print("sample", seed, sample_decode(toy_next_logits, BOS, EOS, 6, temperature=1.5, seed=seed))

requests = 10_000
for output_tokens in [16, 32, 64]:
    decoder_calls = requests * output_tokens
    print(f"output={output_tokens:>2}: decoder steps={decoder_calls:,}")

### 予測

同じ件数・同じモデルで `max_new_tokens` を32から64へ増やし、毎回上限まで生成した場合、decoder step 数はどうなりますか。

A. 同じ　B. 約2倍　C. 約4倍

分類は通常1回の forward ですが、自己回帰生成は出力長ぶん繰り返します。

In [ ]:
# STEP 19: 予測の処理を実行し、出力を照合する
prediction_4 = "B"
print("答え:", prediction_4, "— 出力長はlatencyと推論費の主要変数です")

### 変える → 書く

`temperature` を0.5と2.0に変え、同じ seed の出力を比べてください。低温は最大候補へ集中し、高温は分布を平らにします。top-k は上位k候補だけ、top-p は累積確率がpに達する最小集合だけを残します。

`greedy_decode` を完成させます。EOS を出した時点で止め、EOS 自体は出力へ含めます。beam search ではこの prefix を複数本保持し、累積 log probability で上位を残します。

In [ ]:
# STEP 20: 変える → 書くの処理を実行し、出力を照合する
def greedy_decode(step_function, start_id, eos_id, max_new_tokens):
    # TODO: argmax tokenをprefixへ足し、EOSまたは上限まで生成する
    return None

learner_greedy = safe_call(lambda: greedy_decode(toy_next_logits, BOS, EOS, 6))

In [ ]:
# STEP 21: 変える → 書くの処理を実行し、出力を照合する
check("greedy tokens", learner_greedy, [1, 2, 3], "各stepのargmaxです")
check("greedy length", safe_call(lambda: len(learner_greedy)), 3, "EOSで早期終了します")
check("greedy EOS", safe_call(lambda: learner_greedy[-1]), 3, "EOS自体は出力に含めます")
check("greedy deterministic", safe_call(lambda: greedy_decode(toy_next_logits, BOS, EOS, 6)), [1, 2, 3], "samplingを使わないので再現します")

## 5. ROUGE と事実照合を分け、方式を選ぶ

ROUGE-N は n-gram の重なり、ROUGE-L は最長共通部分列 (LCS) の長さから precision / recall / F1 を作ります。LCS は AtCoder で使う2次元 DP そのものです。

ただし ROUGE は意味や事実性を直接見ません。docstring の**同じ要約出力**でも、抽象型参照では ROUGE-2 `0.0237`、抽出型参照では `0.6011` と約25倍違いました。これはコードのバグではなく、言い換えを評価できない ROUGE の限界です。

生成要約では数値・ブランド・型番を原文と照合し、unsupported fact を別指標で監視します。

In [ ]:
# STEP 22: 5. ROUGE と事実照合を分け、方式を選ぶの処理を実行し、出力を照合する
def ngrams(tokens, n):
    return [tuple(tokens[index:index + n]) for index in range(len(tokens) - n + 1)]

def rouge_n_recall(reference_tokens, candidate_tokens, n):
    reference_counts = Counter(ngrams(reference_tokens, n))
    candidate_counts = Counter(ngrams(candidate_tokens, n))
    overlap = sum((reference_counts & candidate_counts).values())
    total = sum(reference_counts.values())
    return overlap / total if total else 0.0

reference_tokens = ["配送", "が", "速い", "電池", "が", "長持ち"]
candidate_tokens = ["配送", "が", "速い", "電池", "が", "持つ"]
print("ROUGE-1 recall:", round(rouge_n_recall(reference_tokens, candidate_tokens, 1), 3))
print("ROUGE-2 recall:", round(rouge_n_recall(reference_tokens, candidate_tokens, 2), 3))

measured_rouge = pd.DataFrame([
    ("中心性top-k", 0.0174, 0.4906),
    ("MMR λ=0.7", 0.0237, 0.6011),
], columns=["method", "ROUGE2_abstract_reference", "ROUGE2_extractive_reference"])
measured_rouge

In [ ]:
# STEP 23: 5. ROUGE と事実照合を分け、方式を選ぶの処理を実行し、出力を照合する
KNOWN_BRANDS = ["アカネ電機", "ソライロ", "ミドリワークス", "ハヤブサ", "ノースピーク"]

def explicit_facts(text):
    numbers = {f"number:{value}" for value in re.findall(r"\d+(?:\.\d+)?", str(text))}
    brands = {f"brand:{brand}" for brand in KNOWN_BRANDS if brand in str(text)}
    return numbers | brands

row = train_df.iloc[0]
extractive_unsupported = explicit_facts(row["extractive_reference"]) - explicit_facts(row["document"])
hallucinated_unsupported = explicit_facts(row["hallucinated_summary"]) - explicit_facts(row["document"])
print("extractive unsupported:", extractive_unsupported)
print("hallucinated unsupported:", hallucinated_unsupported)

def monthly_api_cost(requests, input_tokens, output_tokens, price_per_million):
    return requests * (input_tokens + output_tokens) / 1_000_000 * price_per_million

# 架空の入力値。見積り時に最新条件を引数へ入れる。
for output_tokens in [32, 64, 128]:
    cost = monthly_api_cost(100_000, 600, output_tokens, 2.0)
    print(f"output={output_tokens:>3} -> illustrative monthly token cost={cost:.2f}")

decision_table = pd.DataFrame([
    ("extractive", "原文内に限定", "低〜中", "最小", "監査しやすい"),
    ("self-hosted generator", "照合が必要", "中〜高", "学習+推論", "運用負荷あり"),
    ("LLM API", "照合が必要", "高い可能性", "token従量", "導入容易・外部依存"),
], columns=["method", "factuality", "quality_ceiling", "cost_shape", "operations"])
decision_table

### 予測

候補要約が参照と同義の言い換えでも、使う語が違う場合 ROUGE はどうなり得ますか。

A. 必ず満点　B. 低くなり得る　C. 必ず0

ROUGE は n-gram / LCS の表面的な重なりを測ります。

In [ ]:
# STEP 24: 予測の処理を実行し、出力を照合する
prediction_5 = "B"
print("答え:", prediction_5, "— 人手評価や事実照合と組み合わせます")

### 変える → 書く

参照と候補の単語順を入れ替え、ROUGE-1 と ROUGE-L の変化を見てください。ROUGE-1 の bag 的な重なりでは見えない順序差を LCS が一部捉えます。

`lcs_length` を完成させます。`dp[i, j]` を「左i要素と右j要素までの LCS 長」とします。文字列でも token list でも動くよう、index と `len` だけを使います。

方式選択では、抽出型 / 自前生成 / API を品質・事実性・latency・運用負荷・最新のコスト入力で比較します。出力長上限を費用だけで下げず、要約品質の制約も同時に置きます。

In [ ]:
# STEP 25: 変える → 書くの処理を実行し、出力を照合する
def lcs_length(left, right):
    # TODO: 2次元DPで最長共通部分列の長さを返す
    return None

learner_lcs = safe_call(lambda: lcs_length("ABCBDAB", "BDCABA"))
learner_lcs_empty = safe_call(lambda: lcs_length("", "ABC"))
learner_lcs_reverse = safe_call(lambda: lcs_length("BDCABA", "ABCBDAB"))

In [ ]:
# STEP 26: 変える → 書くの処理を実行し、出力を照合する
check("LCS length", learner_lcs, 4, "DPの右下が答えです")
check("LCS empty", learner_lcs_empty, 0, "片方が空なら0です")
check("LCS symmetry", learner_lcs_reverse, 4, "左右を交換しても長さは同じです")
check("ROUGE-L recall", safe_call(lambda: learner_lcs / len("BDCABA")), 4 / 6, "LCS / reference length")

<!-- REAL_T5_SECTION_UNIT11 -->
## 実ライブラリで確認: 極小 T5 の teacher forcing と生成

NumPy の shift-right と greedy decode で部品を理解したら、`T5ForConditionalGeneration` で同じ流れをつなぎます。`T5Config` は C# の設定オブジェクト、モデルは encoder と decoder を内包する実装です。

`labels` を渡すだけで、モデル内部が decoder 入力の右シフトと cross entropy を行います。検証用セルはランダム初期化なので意味のある日本語は生成しませんが、事前学習済み重みをダウンロードせず API と shape を本物で確認できます。

`d_model` はtoken表現の幅、`d_ff` は各層内のfeed-forward幅、`num_heads` はattentionを並列に見る本数です。token ID は文字そのものではなく語彙表の整数indexです。forwardの `loss` は正解tokenとの誤差、`logits` は各位置・各語彙IDの生スコアです。

In [ ]:
# STEP 1: deviceを1か所で選び、極小T5とbatchを同じ場所へ置く
import torch
from transformers import T5Config, T5ForConditionalGeneration

torch.set_num_threads(1)
torch.manual_seed(11)
tiny_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tiny_t5_config = T5Config(
    vocab_size=32, d_model=32, d_ff=64,
    num_layers=1, num_decoder_layers=1,
    num_heads=2, d_kv=16, dropout_rate=0.0,
    pad_token_id=0, decoder_start_token_id=0, eos_token_id=1,
)
tiny_t5 = T5ForConditionalGeneration(tiny_t5_config).to(tiny_device)
tiny_batch = {
    "input_ids": torch.tensor([[5, 6, 1, 0], [7, 8, 1, 0]], device=tiny_device),
    "attention_mask": torch.tensor([[1, 1, 1, 0], [1, 1, 1, 0]], device=tiny_device),
    "labels": torch.tensor([[9, 10, 1, -100], [11, 12, 1, -100]], device=tiny_device),
}

# STEP 2: labels付きforward→backward→更新を2step通す
tiny_optimizer = torch.optim.AdamW(tiny_t5.parameters(), lr=0.01)
tiny_losses = []
tiny_t5.train()
for _ in range(2):
    tiny_optimizer.zero_grad()
    tiny_output = tiny_t5(**tiny_batch)
    tiny_output.loss.backward()
    tiny_optimizer.step()
    tiny_losses.append(float(tiny_output.loss.detach()))

# STEP 3: 同じdeviceの入力をgenerateへ渡す
tiny_generated = tiny_t5.generate(
    input_ids=tiny_batch["input_ids"],
    attention_mask=tiny_batch["attention_mask"],
    max_new_tokens=3,
)
print("device:", tiny_device)
print("losses:", [round(value, 4) for value in tiny_losses])
print("logits shape:", tuple(tiny_output.logits.shape))
print("generated shape:", tuple(tiny_generated.shape))

## 振り返り

次を自分の言葉で説明してください。

1. similarity 行列が対称で対角1になる理由
2. MMR の λ を大きく / 小さくしたときの変化
3. teacher forcing の decoder 入力と labels が1位置ずれる理由
4. greedy / beam / sampling の品質・再現性・コスト差
5. ROUGE が高くても fact check が必要な理由

特に、docstring の低い抽象型 ROUGE は実装不良ではありません。参照の言い換え方に依存する評価指標の限界です。

## まとめ — 要約方式を根拠つきで選ぶ

- 文分割 → TF-IDF / 埋め込み → cosine で文 graph を作る
- PageRank 風中心性で relevance、MMR で非冗長性を加える
- encoder-decoder は入力 context を参照し、出力を自己回帰で書く
- teacher forcing は shift right、pad は `-100` で loss から除外
- greedy / beam / sampling は速度・探索・多様性の交換条件
- ROUGE-N / LCS の ROUGE-L と fact 照合を分けて監視する
- まず抽出型、必要な品質差があるとき自前生成 / API を最新条件で比較する
- 生成費用と latency は output token 長に概ね比例する

全チェックが `OK` になったら、ex01〜ex04 で中心性・MMR・ROUGE・teacher forcing・デコードを自力実装し、最後に品質・事実性・コストを含む要約パイプラインを完成させます。